# 4a단계: 자동 ROI (SIFT 특징 매칭 + 호모그래피)

지금까지 ROI(경기장 영역)는 클립마다 직접 좌표를 정했어요. 같은 경기장이라면 **ROI를 아는 프레임(참조)**과 새 프레임 사이의
원근 변환(호모그래피)을 구해서 ROI를 자동으로 옮길 수 있어요. 이 노트북은 그 방법이 직접 정한 ROI를 대신할 수 있는지 검증합니다.

**실행 순서:** 위에서부터 끝까지 (직접 입력할 값 없음)

## 공통 설정

In [ ]:
import os, json, glob, shutil, zipfile, re
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from google.colab import drive
drive.mount('/content/drive')

# 한글 글꼴 (그래프의 한글이 네모로 깨지지 않게)
import subprocess, matplotlib.font_manager as fm
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run("apt-get -qq install -y fonts-nanum > /dev/null", shell=True)
if os.path.exists(_font):
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
else:
    print("⚠️ 한글 글꼴 설치 실패: 그래프의 한글이 네모로 보일 수 있어요")
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# ===== 경로 (v3 노트북 Cell 2와 동일하게) =====
FOLDER_PATH      = "/content/drive/MyDrive/2025년 홍익 vs 국민 드론뷰"
DRIVE_OUTPUT_DIR = os.path.join(FOLDER_PATH, "pipeline_outputs")   # v3의 tracking_final.json 위치
EVAL_DIR         = os.path.join(DRIVE_OUTPUT_DIR, "eval")           # 이번에 새로 만드는 평가 폴더
IMG_DIR          = os.path.join(EVAL_DIR, "images")                 # 평가용 프레임 20장 (정답 라벨의 기준 이미지)
PRE_DIR          = os.path.join(EVAL_DIR, "prelabels")              # 사전 라벨
for _d in (EVAL_DIR, IMG_DIR, PRE_DIR):
    os.makedirs(_d, exist_ok=True)

# ===== 클래스 =====
# 정답(GT)에는 0~2만 남아야 합니다. 3(Unknown)은 "라벨러가 확인해야 할 박스" 표시용입니다.
CLASS_NAMES = ["Hongik", "Kookmin", "Referee", "Unknown"]
CLS_ID = {n: i for i, n in enumerate(CLASS_NAMES)}

# ===== 평가 클립 =====
# play_range : 득점 플레이 구간 (v3 frame_id 기준, 1부터 시작, 양 끝 포함)
# scorer     : v3에서 득점 선수 경로를 만들 때 쓴 (track_id, 프레임 구간). 득점 선수 '정답 위치'로 사용.
#              ⚠️ 0156의 #171, #183 구간은 v3 노트북 주석에 '추정, 확인 필요'로 되어 있었습니다.
#                 v3의 프레임 브라우저 셀로 확인 후 틀렸다면 여기 숫자를 고치세요.
EVAL_CLIPS = {
    "DJI_20250913133612_0144_D": {
        "clip_key": 34,
        "play_range": (413, 649),
        "scorer": [(2122, (413, 649))],
    },
    "DJI_20250913135020_0156_D": {
        "clip_key": 44,
        "play_range": (167, 525),
        "scorer": [(184, (167, 214)), (171, (215, 313)), (183, (314, 393)), (208, (410, 525))],
    },
}
FRAMES_PER_CLIP = 10

# ===== 매칭 기준 =====
# 선수가 10~25px로 작아서 IoU 0.5는 박스가 조금만 어긋나도 오답이 됩니다.
# 대신 '박스 중심 거리'로 짝을 짓습니다: 거리 <= max(MIN_PX, RATIO × 정답 박스 긴 변)
MATCH_MIN_PX = 10
MATCH_RATIO  = 0.5
SCORER_MATCH_PX = 15   # 득점 선수 정답 위치(발)와 예측 박스 발 위치가 이 거리(px) 이내면 '찾았다'


def video_path_of(stem):
    for ext in (".MP4", ".mp4", ".MOV", ".mov"):
        p = os.path.join(FOLDER_PATH, stem + ext)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"영상을 찾을 수 없습니다: {stem}")

def tracking_json_of(stem, folder=DRIVE_OUTPUT_DIR):
    return os.path.join(folder, f"{stem}_tracking_final.json")

for stem in EVAL_CLIPS:
    print(stem, "| 영상:", os.path.exists(video_path_of(stem)), "| v3 JSON:", os.path.exists(tracking_json_of(stem)))

In [ ]:
# ===== 공통 함수 =====

def load_tracking(json_path):
    """v3 형식 JSON → {frame_id: [player dict, ...]}, field_polygon"""
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    by_frame = {fr["frame_id"]: fr["players"] for fr in data["frames"]}
    poly = np.array(data.get("field_polygon", []), dtype=np.int32)
    return by_frame, poly


def read_frames_sequential(video_path, frame_ids):
    """
    frame_id(1부터 시작)에 해당하는 프레임을 '처음부터 순서대로 읽어서' 가져옵니다.
    cap.set(POS_FRAMES)로 바로 점프하면 드론 영상(H.264/H.265)에서 몇 프레임씩 어긋날 수 있어서,
    v3가 프레임을 센 방식(처음부터 1, 2, 3...)과 똑같이 맞추기 위해 순차로 읽습니다.
    """
    wanted = set(frame_ids)
    last = max(wanted)
    out = {}
    cap = cv2.VideoCapture(video_path)
    fid = 0
    while fid < last:
        ok, frame = cap.read()
        if not ok:
            break
        fid += 1
        if fid in wanted:
            out[fid] = frame
    cap.release()
    missing = wanted - set(out)
    if missing:
        print(f"⚠️ 읽지 못한 프레임: {sorted(missing)}")
    return out


def pick_frames(play_range, n):
    a, b = play_range
    return sorted(set(np.linspace(a, b, n).round().astype(int).tolist()))


def inside(poly, x, y):
    if poly is None or len(poly) < 3:
        return True
    return cv2.pointPolygonTest(poly.reshape(-1, 1, 2), (float(x), float(y)), False) >= 0


def xyxy_to_yolo_line(cls_id, box, W, H):
    x1, y1, x2, y2 = box
    x1, x2 = max(0, x1), min(W, x2)
    y1, y2 = max(0, y1), min(H, y2)
    cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
    w, h = (x2 - x1) / W, (y2 - y1) / H
    return f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"


def yolo_line_to_xyxy(line, W, H):
    c, cx, cy, w, h = line.split()[:5]
    cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
    return int(c), [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]


def center(b):
    return ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)


def box_iou(a, b):
    """Day 2에서 만든 IoU (xyxy)"""
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def match_by_center(gt_boxes, pred_boxes):
    """
    정답 박스와 예측 박스를 1:1로 짝짓기 (헝가리안 알고리즘, 중심 거리 기준).
    return: [(gi, pi), ...], 짝 없는 정답 인덱스, 짝 없는 예측 인덱스
    """
    G, P = len(gt_boxes), len(pred_boxes)
    if G == 0 or P == 0:
        return [], list(range(G)), list(range(P))
    gc = np.array([center(b) for b in gt_boxes])
    pc = np.array([center(b) for b in pred_boxes])
    dist = np.linalg.norm(gc[:, None, :] - pc[None, :, :], axis=2)          # (G, P)
    gsize = np.array([max(b[2] - b[0], b[3] - b[1]) for b in gt_boxes])
    thr = np.maximum(MATCH_MIN_PX, MATCH_RATIO * gsize)[:, None]            # 정답 박스별 허용 거리
    cost = np.where(dist <= thr, dist, 1e6)
    gi, pi = linear_sum_assignment(cost)
    pairs = [(g, p) for g, p in zip(gi, pi) if cost[g, p] < 1e6]
    mg = {g for g, _ in pairs}
    mp = {p for _, p in pairs}
    return pairs, [g for g in range(G) if g not in mg], [p for p in range(P) if p not in mp]


print("✅ 공통 함수 준비 완료")

## 평가 준비 (0단계 파트 B와 같은 코드)

In [ ]:
GT_ZIP = os.path.join(EVAL_DIR, "gt_roboflow.zip")
GT_DIR = os.path.join("/content", "gt_extracted")
shutil.rmtree(GT_DIR, ignore_errors=True)
with zipfile.ZipFile(GT_ZIP) as z:
    z.extractall(GT_DIR)


def original_name(fname):
    """Roboflow가 바꾼 파일명 → 원래 이름. 예: 'DJI_..._f00413_jpg.rf.ab12cd.txt' → 'DJI_..._f00413'"""
    base = os.path.basename(fname)
    base = re.split(r"_(?:jpg|jpeg|png)\.rf\.", base, flags=re.IGNORECASE)[0]
    return os.path.splitext(base)[0]


def read_names_yaml(gt_dir):
    """Roboflow data.yaml의 클래스 순서를 읽음 (Roboflow가 순서를 바꿀 수 있어서 이름으로 다시 매핑)"""
    ymls = glob.glob(os.path.join(gt_dir, "**", "data.yaml"), recursive=True)
    if not ymls:
        return CLASS_NAMES
    txt = open(ymls[0], encoding="utf-8").read()
    m = re.search(r"names:\s*\[(.*?)\]", txt, re.S)
    if m:
        return [s.strip().strip("'\"") for s in m.group(1).split(",") if s.strip()]
    return re.findall(r"^\s*-\s*['\"]?([^'\"\n]+?)['\"]?\s*$", txt.split("names:")[1], re.M)


def load_gt(gt_dir):
    """→ {name: {"W","H","boxes":[(class_name, xyxy), ...]}}"""
    rf_names = read_names_yaml(gt_dir)
    gt = {}
    for lbl in glob.glob(os.path.join(gt_dir, "**", "labels", "*.txt"), recursive=True):
        name = original_name(lbl)
        img = cv2.imread(os.path.join(IMG_DIR, name + ".jpg"))   # 원본 해상도 기준으로 복원
        if img is None:
            print("⚠️ 원본 이미지를 찾을 수 없음:", name); continue
        H, W = img.shape[:2]
        boxes = []
        for line in open(lbl).read().strip().splitlines():
            if not line.strip():
                continue
            c, b = yolo_line_to_xyxy(line, W, H)
            boxes.append((rf_names[c], b))
        gt[name] = {"W": W, "H": H, "boxes": boxes}
    return gt


GT = load_gt(GT_DIR)
_cnt = Counter(c for v in GT.values() for c, _ in v["boxes"])
print(f"정답 이미지 {len(GT)}장, 박스 {sum(_cnt.values())}개 → {dict(_cnt)}")
if _cnt.get("Unknown"):
    print(f"⚠️ 정답에 Unknown이 {_cnt['Unknown']}개 남아 있습니다. 탐지 평가에는 포함되지만 팀 정확도에서는 제외됩니다.")

In [ ]:
EVAL_TEAM = ["Hongik", "Kookmin", "Referee"]


def detection_and_team_metrics(run_json_by_stem):
    """정답 vs 예측: 탐지(클래스 무관) P/R/F1 + 짝지어진 박스의 클래스 정확도 + 혼동행렬"""
    tp = fp = fn = 0
    conf_mat = Counter()                     # (정답 클래스, 예측 클래스)
    per_frame = []
    for stem, jpath in run_json_by_stem.items():
        by_frame, _ = load_tracking(jpath)
        for name, g in GT.items():
            if not name.startswith(stem):
                continue
            fid = int(name.split("_f")[-1])
            preds = by_frame.get(fid, [])
            gb = [b for _, b in g["boxes"]]
            pb = [p["bbox"] for p in preds]
            pairs, ug, up = match_by_center(gb, pb)
            tp += len(pairs); fn += len(ug); fp += len(up)
            for gi, pi in pairs:
                conf_mat[(g["boxes"][gi][0], preds[pi]["type"])] += 1
            per_frame.append({"image": name, "gt": len(gb), "pred": len(pb), "TP": len(pairs), "FN": len(ug), "FP": len(up)})
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F1 = 2 * P * R / (P + R) if P + R else 0.0
    team_pairs = {k: v for k, v in conf_mat.items() if k[0] in EVAL_TEAM}
    correct = sum(v for (g, p), v in team_pairs.items() if g == p)
    team_acc = correct / sum(team_pairs.values()) if team_pairs else 0.0
    ref_gt = sum(1 for v in GT.values() for c, _ in v["boxes"] if c == "Referee")
    ref_hit = conf_mat.get(("Referee", "Referee"), 0)
    return {
        "det_precision": round(P, 3), "det_recall": round(R, 3), "det_f1": round(F1, 3),
        "TP": tp, "FP": fp, "FN": fn,
        "team_acc": round(team_acc, 3),
        "referee_recall": round(ref_hit / ref_gt, 3) if ref_gt else None,
    }, conf_mat, pd.DataFrame(per_frame)


def scorer_track_metrics(stem, run_json, cfg, verbose=True):
    """
    득점 선수 추적 품질 (라벨링 없이 측정).
    - 정답 위치: v3에서 사람이 확인해 이어붙인 득점 선수의 발 위치 (cfg["scorer"])
    - 평가할 결과(run_json)에서 매 프레임 그 위치에 가장 가까운 박스의 track_id를 읽음

    지표
    - scorer_ids / scorer_id_switches : '정답 위치가 확실한 프레임'에서만 계산 (믿을 수 있는 값)
    - scorer_coverage : v3가 놓친 구간은 앞뒤 위치를 직선으로 이어 정답 위치를 '추정'해서 포함
                        → 그 구간에서는 근처의 다른 선수가 잡힐 수 있으니 참고용(근사값)
    """
    ref_by_frame, _ = load_tracking(tracking_json_of(stem))       # 정답 경로는 항상 v3 기준
    run_by_frame, _ = load_tracking(run_json)
    known = {}
    for tid, (a, b) in cfg["scorer"]:
        for fid in range(a, b + 1):
            for p in ref_by_frame.get(fid, []):
                if p["track_id"] == tid:
                    known[fid] = p["feet"]
    if not known:
        print(f"⚠️ {stem}: v3 JSON에서 득점 선수 track_id를 찾지 못했습니다. EVAL_CLIPS의 scorer 값을 확인하세요.")
        return {"scorer_frames": 0, "scorer_coverage": 0.0, "scorer_ids": 0, "scorer_id_switches": 0}
    span = range(min(known), max(known) + 1)
    kf = np.array(sorted(known))
    kx = np.array([known[f][0] for f in kf]); ky = np.array([known[f][1] for f in kf])

    seq = []   # (frame, 찾은 track_id 또는 None, 정답 위치가 확실한지)
    for fid in span:
        gx, gy = (known[fid] if fid in known else (np.interp(fid, kf, kx), np.interp(fid, kf, ky)))
        best, bd = None, 1e9
        for p in run_by_frame.get(fid, []):
            d = np.hypot(p["feet"][0] - gx, p["feet"][1] - gy)
            if d < bd:
                best, bd = p, d
        tid = best["track_id"] if best is not None and bd <= SCORER_MATCH_PX else None
        seq.append((fid, tid, fid in known))

    sure = [t for _, t, k in seq if k and t is not None]
    switches = sum(1 for a, b in zip(sure, sure[1:]) if a != b)
    covered = sum(1 for _, t, _ in seq if t is not None)

    if verbose:   # 어느 ID가 어느 구간을 맡았는지 (v3 노트북의 SEGMENTS와 같은 형태) → 영상으로 확인할 때 사용
        runs, cur = [], None
        for fid, t, _ in seq:
            if cur and cur[0] == t and cur[2] == fid - 1:
                cur[2] = fid
            else:
                cur = [t, fid, fid]; runs.append(cur)
        desc = ", ".join(f"{'없음' if t is None else '#' + str(t)}({a}~{b})" for t, a, b in runs)
        print(f"  [{stem}] 득점 선수 경로: {desc}")

    return {
        "scorer_frames": len(seq),
        "scorer_coverage": round(covered / len(seq), 3),
        "scorer_ids": len(set(sure)),
        "scorer_id_switches": switches,
    }


def segment_stats(stem, run_json, cfg):
    """플레이 구간 안의 고유 ID 수, Unknown 비율 (라벨링 없이 측정)"""
    by_frame, _ = load_tracking(run_json)
    a, b = cfg["play_range"]
    ids, types = set(), Counter()
    for fid in range(a, b + 1):
        for p in by_frame.get(fid, []):
            ids.add(p["track_id"]); types[p["type"]] += 1
    total = sum(types.values())
    return {"segment_unique_ids": len(ids),
            "segment_unknown_ratio": round(types["Unknown"] / total, 3) if total else 0.0}


RESULTS_CSV = os.path.join(EVAL_DIR, "eval_results.csv")


def evaluate_run(run_name, run_json_by_stem, note=""):
    """
    한 번의 실험 결과를 채점해서 eval_results.csv에 한 줄 추가합니다.
    run_json_by_stem: {stem: 그 실험이 만든 tracking_final.json 경로}
    """
    det, conf_mat, per_frame = detection_and_team_metrics(run_json_by_stem)
    row = {"run": run_name, "note": note, **det}
    for stem, jpath in run_json_by_stem.items():
        cfg = EVAL_CLIPS[stem]
        k = cfg["clip_key"]
        for key, val in {**scorer_track_metrics(stem, jpath, cfg), **segment_stats(stem, jpath, cfg)}.items():
            row[f"{key}_{k}"] = val
    df_new = pd.DataFrame([row])
    if os.path.exists(RESULTS_CSV):
        old = pd.read_csv(RESULTS_CSV)
        old = old[old["run"] != run_name]                      # 같은 이름으로 다시 돌리면 덮어쓰기
        df_new = pd.concat([old, df_new], ignore_index=True)
    df_new.to_csv(RESULTS_CSV, index=False, encoding="utf-8-sig")
    return row, conf_mat, per_frame


def show_confusion(conf_mat):
    rows = EVAL_TEAM + (["Unknown"] if any(g == "Unknown" for g, _ in conf_mat) else [])
    cols = CLASS_NAMES
    return pd.DataFrame([[conf_mat.get((r, c), 0) for c in cols] for r in rows],
                        index=[f"정답:{r}" for r in rows], columns=[f"예측:{c}" for c in cols])

print("✅ 평가 함수 준비 완료")

In [ ]:
# 틀린 곳을 눈으로 확인: 초록=맞게 찾음(TP), 빨강=놓친 정답(FN), 파랑=잘못 잡음(FP)
def show_errors(name, run_json_by_stem):
    stem = name.split("_f")[0]
    fid = int(name.split("_f")[-1])
    by_frame, poly = load_tracking(run_json_by_stem[stem])
    preds = by_frame.get(fid, [])
    g = GT[name]
    pairs, ug, up = match_by_center([b for _, b in g["boxes"]], [p["bbox"] for p in preds])
    img = cv2.imread(os.path.join(IMG_DIR, name + ".jpg"))
    cv2.polylines(img, [poly.reshape(-1, 1, 2)], True, (0, 255, 0), 1)
    for gi, pi in pairs:
        b = preds[pi]["bbox"]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 200, 0), 2)
    for gi in ug:
        b = g["boxes"][gi][1]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 0, 255), 2)
    for pi in up:
        b = preds[pi]["bbox"]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (255, 0, 0), 2)
    plt.figure(figsize=(16, 9)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"{name}  TP={len(pairs)}  FN={len(ug)}  FP={len(up)}"); plt.axis("off"); plt.show()

In [ ]:
# ===== 자동 ROI: 참조 프레임 → 새 프레임으로 경기장 영역 옮기기 =====
# 같은 경기장이면 '경기장 영역을 이미 아는 프레임(참조)'과 새 프레임은 같은 평면(경기장)을 다른 위치·각도에서 본 것.
# 평면 → 평면의 원근 변환은 3×3 호모그래피 H 하나로 표현됨:  [x', y', w']ᵀ = H [x, y, 1]ᵀ,  (x'/w', y'/w')
#   1) SIFT 특징점 추출 (크기·회전·밝기 변화에 강함) — 반 해상도에서 계산해 속도 확보
#   2) 참조 ↔ 새 프레임 특징점 짝짓기 + Lowe 비율 검사(0.75)로 애매한 짝 제거
#   3) RANSAC으로 호모그래피 추정: 무작위 4쌍으로 H를 만들고, 5px 안에 맞는 짝(인라이어)이 가장 많은 H 선택
#      → 나무·자동차처럼 경기장 평면이 아닌 점, 잘못된 짝은 자동으로 버려짐
#   4) 참조 ROI 다각형을 H로 옮김. 참조 화면 가장자리에서 잘린 변은 실제 라인 방향으로 화면 밖까지 연장한 뒤 옮김
#      (새 프레임이 더 넓게 찍혔을 때 ROI가 참조 화면 크기에서 잘리지 않도록)
#   5) 인라이어가 MIN_INLIERS 미만이면 '경기장의 다른 쪽을 찍은 영상'으로 보고 실패 처리 (수동 입력 필요)
import time
SIFT = cv2.SIFT_create(nfeatures=6000)
BF = cv2.BFMatcher()
FEAT_SCALE = 0.5
MIN_INLIERS = 100
RATIO = 0.75


def sift_features(img):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, None, fx=FEAT_SCALE, fy=FEAT_SCALE, interpolation=cv2.INTER_AREA)
    kp, des = SIFT.detectAndCompute(g, None)
    return np.float32([k.pt for k in kp]).reshape(-1, 2) / FEAT_SCALE, des


def _border_sides(p, W, H, tol=3):
    x, y = p
    return {s for s, c in (("L", x <= tol), ("R", x >= W - 1 - tol), ("T", y <= tol), ("B", y >= H - 1 - tol)) if c}


def open_polygon(poly, W, H, L=2500):
    """화면 가장자리에서 잘린 ROI 꼭짓점을 실제 라인 방향으로 L px 연장 (화면 모서리 꼭짓점은 버림)"""
    P = [np.float64(p) for p in poly]; out = []
    for i, p in enumerate(P):
        a, b = P[i - 1], P[(i + 1) % len(P)]
        bp = _border_sides(p, W, H)
        if not bp:
            out.append(p); continue
        ea, eb = bool(bp & _border_sides(a, W, H)), bool(bp & _border_sides(b, W, H))   # 이웃과 같은 가장자리 = 화면 경계 변
        if ea and eb:
            continue
        q = b if ea else a                                                              # 실제 라인 쪽 이웃
        d = (p - q) / (np.linalg.norm(p - q) + 1e-9)
        out.append(p + d * L)
    return np.array(out)


def make_ref(name, img, roi_pts):
    H, W = img.shape[:2]
    pts, des = sift_features(img)
    return {"name": name, "pts": pts, "des": des, "poly_open": open_polygon(roi_pts, W, H), "roi": np.asarray(roi_pts)}


def estimate_transform(ref, pts, des, model="homography"):
    """참조 → 새 프레임 변환과 인라이어 수. model: 'homography'(원근, 8자유도) / 'similarity'(이동·회전·크기, 4자유도)"""
    if des is None or ref["des"] is None or len(des) < 10:
        return None, 0
    good = [p[0] for p in BF.knnMatch(ref["des"], des, k=2) if len(p) == 2 and p[0].distance < RATIO * p[1].distance]
    if len(good) < 10:
        return None, 0
    a = ref["pts"][[m.queryIdx for m in good]]; b = pts[[m.trainIdx for m in good]]
    if model == "homography":
        Hm, inl = cv2.findHomography(a, b, cv2.RANSAC, 5.0)
    else:
        A, inl = cv2.estimateAffinePartial2D(a, b, method=cv2.RANSAC, ransacReprojThreshold=5.0)
        Hm = None if A is None else np.vstack([A, [0, 0, 1]])
    return (None, 0) if Hm is None else (Hm, int(inl.sum()))


def warp_roi_mask(ref, Hm, W, H):
    pts = np.hstack([ref["poly_open"], np.ones((len(ref["poly_open"]), 1))]) @ Hm.T
    if np.any(pts[:, 2] <= 1e-6):                       # 연장한 점이 지평선 너머로 넘어감 → 실패
        return None
    m = np.zeros((H, W), np.uint8)
    cv2.fillPoly(m, [np.round(pts[:, :2] / pts[:, 2:]).astype(np.int32).reshape(-1, 1, 2)], 1)
    return m


def mask_to_poly(m):
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return cv2.approxPolyDP(max(cnts, key=cv2.contourArea), 2.0, True).reshape(-1, 2)


def auto_roi(img, refs, model="homography", exclude=()):
    """→ (ROI 다각형 또는 None, 고른 참조 이름, 인라이어 수)"""
    H, W = img.shape[:2]
    pts, des = sift_features(img)
    best = None
    for ref in refs:
        if ref["name"] in exclude:
            continue
        Hm, n = estimate_transform(ref, pts, des, model)
        if Hm is not None and n >= MIN_INLIERS and (best is None or n > best[2]):
            best = (ref, Hm, n)
    if best is None:
        return None, None, 0
    m = warp_roi_mask(best[0], best[1], W, H)
    if m is None or m.sum() < 0.05 * W * H:             # 화면의 5%도 안 되면 잘못 옮긴 것으로 봄
        return None, best[0]["name"], best[2]
    return mask_to_poly(m), best[0]["name"], best[2]


def expand_polygon(poly, margin, W, H):
    """다각형을 margin(px)만큼 바깥으로 넓힘 (1a와 같은 방법: 채운 마스크 팽창 → 다시 다각형)"""
    mask = np.zeros((H, W), np.uint8)
    cv2.fillPoly(mask, [np.asarray(poly, np.int32).reshape(-1, 1, 2)], 255)
    if margin > 0:
        mask = cv2.dilate(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * margin + 1, 2 * margin + 1)))
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return cv2.approxPolyDP(max(cnts, key=cv2.contourArea), 2.0, True).reshape(-1, 2)


def poly_iou(p1, p2, W, H):
    a = np.zeros((H, W), np.uint8); b = np.zeros((H, W), np.uint8)
    cv2.fillPoly(a, [np.int32(p1).reshape(-1, 1, 2)], 1); cv2.fillPoly(b, [np.int32(p2).reshape(-1, 1, 2)], 1)
    return float((a & b).sum() / max(1, (a | b).sum()))

print("✅ 자동 ROI 함수 준비 완료 (OpenCV", cv2.__version__, ")")

---
# 4a: 자동 ROI 검증

| 순서 | 내용 |
|---|---|
| ① | ROI를 직접 정했던 프레임 4개(0144 · 0156 · 0143 · 0157)를 **참조**로 등록 |
| ② | **하나 빼고 검증**: 각 클립의 ROI를 *나머지 3개 참조만으로* 자동 계산 → 직접 정한 ROI와 겹침(IoU) 비교. 호모그래피 vs 유사 변환 |
| ③ | **실전 검증**: 평가 클립(0144 · 0156)은 *튜닝 클립 참조만으로* 30프레임마다 ROI를 자동 계산 → v3 · v5 결과에 적용해 채점 → 직접 정한 ROI 결과와 비교 |

**런타임: CPU로 충분** · 약 5분

In [ ]:
# ① 참조 프레임 (ROI를 직접 정했던 바로 그 프레임)
KNOWN = {
    "DJI_20250913133612_0144_D": (413, [[245, 530], [1860, 543], [1919, 776], [1919, 1079], [107, 1079]]),
    "DJI_20250913135020_0156_D": (167, [[0, 0], [1632, 0], [1712, 810], [0, 787]]),
    "DJI_20250913133530_0143_D": (1, [[270, 459], [1880, 469], [1919, 612], [1919, 1079], [119, 1079]]),
    "DJI_20250913135111_0157_D": (1, [[0, 0], [1675, 0], [1761, 737], [0, 733]]),
}
ROI_MARGIN_PX = 5
ROI_UPDATE_EVERY = 30          # 실행 노트북과 같게: 30프레임(1초)마다 ROI 다시 계산
short = lambda s: s[-6:-2]


def read_frame(stem, fid):
    cap = cv2.VideoCapture(video_path_of(stem)); img = None
    for _ in range(fid):
        ok, img = cap.read()
        if not ok:
            raise ValueError(f"{stem}: {fid}프레임을 읽을 수 없습니다")
    cap.release()
    return img

REF_IMG = {s: read_frame(s, f) for s, (f, _) in KNOWN.items()}
REFS = [make_ref(s, REF_IMG[s], roi) for s, (_, roi) in KNOWN.items()]
for r in REFS:
    print(f"참조 {short(r['name'])}: SIFT 특징점 {len(r['pts'])}개")

In [ ]:
# ② 하나 빼고 검증: 자기 자신을 뺀 참조 3개로 ROI 계산
rows, AUTO_LOO = [], {}
for s, (f, roi) in KNOWN.items():
    img = REF_IMG[s]; H, W = img.shape[:2]
    row = {"클립": short(s)}
    for model, name in [("homography", "호모그래피"), ("similarity", "유사 변환")]:
        t0 = time.time()
        poly, ref, n = auto_roi(img, REFS, model, exclude=(s,))
        row[f"{name} IoU"] = round(poly_iou(poly, roi, W, H), 4) if poly is not None else None
        row[f"{name} 참조"] = short(ref) if ref else "실패"
        row[f"{name} 인라이어"] = n
        if model == "homography":
            AUTO_LOO[s] = poly; row["시간(초)"] = round(time.time() - t0, 2)
    rows.append(row)
LOO_DF = pd.DataFrame(rows).set_index("클립")
display(LOO_DF)
print("IoU 1.0 = 직접 정한 ROI와 완전히 같음. 0.99 = 경계가 몇 px 차이")

fig, axes = plt.subplots(2, 2, figsize=(16, 9.5))
for ax, (s, (f, roi)) in zip(axes.ravel(), KNOWN.items()):
    vis = REF_IMG[s].copy()
    cv2.polylines(vis, [np.int32(roi).reshape(-1, 1, 2)], True, (0, 255, 0), 6)
    if AUTO_LOO[s] is not None:
        cv2.polylines(vis, [np.int32(AUTO_LOO[s]).reshape(-1, 1, 2)], True, (0, 0, 255), 3)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); ax.axis("off")
    ax.set_title(f"{short(s)} · 초록=직접 정한 ROI, 빨강=자동 ROI (IoU {LOO_DF.loc[short(s), '호모그래피 IoU']})")
plt.tight_layout(); plt.show()

In [ ]:
# ③ 실전 검증: 평가 클립 ROI를 튜닝 클립 참조만으로 30프레임마다 자동 계산
EXCLUDE_EVAL = tuple(EVAL_CLIPS)


def auto_roi_track(stem, last_fid):
    """1 ~ last_fid 프레임을 순서대로 읽으며 ROI_UPDATE_EVERY마다 자동 ROI. → {frame_id: 다각형(여유 포함)}, 기록"""
    cap = cv2.VideoCapture(video_path_of(stem))
    roi_at, log, cur, fid = {}, [], None, 0
    while fid < last_fid:
        ok, img = cap.read()
        if not ok:
            break
        fid += 1
        if (fid - 1) % ROI_UPDATE_EVERY == 0:
            poly, ref, n = auto_roi(img, REFS, "homography", exclude=EXCLUDE_EVAL)
            H, W = img.shape[:2]
            if poly is not None:
                cur = expand_polygon(poly, ROI_MARGIN_PX, W, H)
            log.append({"frame": fid, "참조": short(ref) if ref else "실패", "인라이어": n,
                        "직접 ROI와 IoU": round(poly_iou(poly, KNOWN[stem][1], W, H), 4) if poly is not None else None})
        roi_at[fid] = cur
    cap.release()
    return roi_at, pd.DataFrame(log)


def apply_roi_to_json(src_json, roi_at, out_json, note):
    data = json.load(open(src_json, encoding="utf-8"))
    removed = 0
    for fr in data["frames"]:
        poly = roi_at.get(fr["frame_id"])
        if poly is None:
            continue
        keep = [p for p in fr["players"] if inside(poly, p["feet"][0], p["feet"][1])]
        removed += len(fr["players"]) - len(keep); fr["players"] = keep
    first = next((p for p in roi_at.values() if p is not None), None)
    if first is not None:
        data["field_polygon"] = np.asarray(first).tolist()
    data.setdefault("notes", []).append(note)
    os.makedirs(os.path.dirname(out_json), exist_ok=True)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    return removed


ROI_AT = {}
for stem, cfg in EVAL_CLIPS.items():
    t0 = time.time()
    ROI_AT[stem], log = auto_roi_track(stem, cfg["play_range"][1])
    print(f"\n{short(stem)}: ROI 계산 {len(log)}번 ({time.time() - t0:.0f}s) · 실패 {int((log['참조'] == '실패').sum())}번 · "
          f"직접 ROI와 IoU 최소 {log['직접 ROI와 IoU'].min()} / 평균 {log['직접 ROI와 IoU'].mean():.4f}")
    display(log.iloc[::max(1, len(log) // 8)])

SOURCES = {"v3_autoroi": (DRIVE_OUTPUT_DIR, "v3 + 자동 ROI (1a의 직접 ROI 대신)"),
           "v5_autoroi": (os.path.join(EVAL_DIR, "runs", "v5_cnn"), "v5 + 자동 ROI (직접 ROI 대신)")}
for run_name, (src_dir, note) in SOURCES.items():
    run = {}
    for stem in EVAL_CLIPS:
        out = os.path.join(EVAL_DIR, "runs", run_name, f"{stem}_tracking_final.json")
        n_rm = apply_roi_to_json(os.path.join(src_dir, f"{stem}_tracking_final.json"), ROI_AT[stem], out, note)
        print(f"[{run_name}] {short(stem)}: ROI 밖이라 지운 박스 {n_rm}개")
        run[stem] = out
    evaluate_run(run_name, run, note=note)

In [ ]:
# 비교: 직접 정한 ROI vs 자동 ROI
res = pd.read_csv(RESULTS_CSV).set_index("run")
COLS = ["det_precision", "det_recall", "det_f1", "TP", "FP", "FN", "team_acc"]
order = [r for r in ["v3_baseline", "step1a_roi", "v3_autoroi", "v5_cnn", "v5_autoroi"] if r in res.index]
display(res.loc[order, COLS])
print("기대: step1a_roi ≈ v3_autoroi, v5_cnn ≈ v5_autoroi → 자동 ROI가 직접 정한 ROI를 대신할 수 있음")

## 결과 공유
1. ② 표와 2×2 그림
2. ③ 출력(클립별 IoU 최소/평균, 지운 박스 수)과 마지막 비교표